In [288]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import os
pd.options.mode.chained_assignment = None
import optuna
import time
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold, train_test_split
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, RobustScaler
import pickle


In [289]:
# Cargamos los csv de los tifs
path = "saved_files/dataset"
depth = "lt_1"
dfs = {}
all_datasets = False
for archivo in os.listdir(path):
    if all_datasets:
        if f"{depth}" in archivo:
            nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
            ruta_completa = os.path.join(path, archivo)
            dfs[nombre_sin_extension] = pd.read_csv(ruta_completa)
    else:
        if archivo.endswith(f"{depth}_features.csv"):
            nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
            ruta_completa = os.path.join(path, archivo)
            dfs[nombre_sin_extension[:-9]] = pd.read_csv(ruta_completa)

In [214]:
dfs.keys()

dict_keys(['C2RCC_rhow_5x5_depth_lt_1', 'C2X-Complex_rhown_3x3_depth_lt_1', 'TOA_9x9_depth_lt_1', 'C2X_rhow_3x3_depth_lt_1', 'C2RCC_rhow_9x9_depth_lt_1', 'C2X-Complex_rhown_5x5_depth_lt_1', 'C2X_rhown_9x9_depth_lt_1', 'TOA_3x3_depth_lt_1', 'C2X-Complex_rhown_9x9_depth_lt_1', 'C2X-Complex_rhow_3x3_depth_lt_1', 'C2X_rhow_1x1_depth_lt_1', 'C2RCC_rhown_1x1_depth_lt_1', 'C2X_rhown_1x1_depth_lt_1', 'C2RCC_rhown_9x9_depth_lt_1', 'C2X_rhow_5x5_depth_lt_1', 'C2X_rhown_5x5_depth_lt_1', 'C2X_rhow_9x9_depth_lt_1', 'C2X_rhown_3x3_depth_lt_1', 'C2RCC_rhown_5x5_depth_lt_1', 'TOA_1x1_depth_lt_1', 'C2X-Complex_rhow_5x5_depth_lt_1', 'C2X-Complex_rhow_9x9_depth_lt_1', 'C2X-Complex_rhow_1x1_depth_lt_1', 'C2X-Complex_rhown_1x1_depth_lt_1', 'C2RCC_rhow_3x3_depth_lt_1', 'TOA_5x5_depth_lt_1', 'C2RCC_rhown_3x3_depth_lt_1', 'C2RCC_rhow_1x1_depth_lt_1'])

In [194]:
dfs["C2RCC_rhow_1x1_depth_lt_1"]

,Date,Buoy,Latitude,Longitude,rhow_B1,rhow_B2,rhow_B3,rhow_B4,rhow_B5,rhow_B6,...,dif_rel_4bands_rhow_B2_B5_B3_B4,dif_rel_4bands_rhow_B2_B5_B4_B3,dif_rel_4bands_rhow_B3_B2_B4_B5,dif_rel_4bands_rhow_B3_B2_B5_B4,dif_rel_4bands_rhow_B3_B4_B5_B2,dif_rel_4bands_rhow_B3_B5_B4_B2,dif_rel_4bands_rhow_B4_B2_B5_B3,dif_rel_4bands_rhow_B4_B3_B5_B2,sum_norm_3bands_rhow_B2_B4_B3,sum_norm_3bands_rhow_B3_B5_B4
0,2016-08-09,CTD1,4187246,695025,0.010973,0.017248,0.033436,0.016780,0.012437,0.003690,...,-0.606,0.885,0.589,1.197,1.272,1.716,0.601,-0.219,0.671,0.914
1,2016-08-09,CTD2,4181518,693105,0.007180,0.010530,0.019375,0.011021,0.008281,0.002545,...,-0.486,0.703,0.509,1.089,0.972,1.293,0.619,-0.218,0.721,0.910
2,2016-08-09,CTD3,4181698,695238,0.007966,0.011565,0.020111,0.010630,0.007752,0.002311,...,-0.400,0.963,0.368,1.010,1.222,1.675,0.534,-0.142,0.701,0.906
3,2016-08-09,CTD4,4180266,698264,0.008062,0.010310,0.014764,0.008631,0.006252,0.001894,...,-0.062,1.064,0.052,0.708,1.104,1.524,0.414,-0.022,0.755,0.898
4,2016-08-09,CTD6,4176009,695829,0.008603,0.013129,0.023304,0.010864,0.007707,0.002229,...,-0.442,1.237,0.365,1.066,1.558,2.196,0.497,-0.121,0.659,0.908
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
446,2023-05-25,CTD8,4174178,693048,0.016512,0.029853,0.043600,0.007112,0.004319,0.001135,...,0.781,6.749,-0.186,0.853,5.986,9.856,0.139,0.018,0.503,0.945
447,2023-05-25,CTD9,4171106,693183,0.007081,0.012752,0.026582,0.009738,0.006888,0.001995,...,-0.879,1.485,0.671,1.377,2.190,3.095,0.505,-0.174,0.572,0.922
448,2023-05-25,CTD10,4170388,695646,0.007636,0.014290,0.026969,0.006672,0.004333,0.001186,...,-0.745,3.050,0.348,1.238,3.739,5.756,0.306,-0.056,0.508,0.930
449,2023-05-25,CTD11,4169609,700351,0.007547,0.014033,0.026831,0.007324,0.004826,0.001328,...,-0.756,2.635,0.394,1.253,3.320,5.038,0.342,-0.071,0.523,0.927


In [29]:
# Para seleccionar manualmente qué dataframes nos quedamos
# Si leemos por profundidad ya no hace falta esto
dfs_to_keep = [
    "C2X_1x1_merge_depth_lt_1", "C2X_3x3_merge_depth_lt_1", "C2X_5x5_merge_depth_lt_1", 
    "C2X-Complex_1x1_merge_depth_lt_1", "C2X-Complex_3x3_merge_depth_lt_1", "C2X-Complex_5x5_merge_depth_lt_1",
    "C2RCC_1x1_merge_depth_lt_1", "C2RCC_3x3_merge_depth_lt_1", "C2RCC_5x5_merge_depth_lt_1",
    "TOA_1x1_merge_depth_lt_1", "TOA_3x3_merge_depth_lt_1", "TOA_5x5_merge_depth_lt_1",
]

#dfs = {k: dfs[k] for k in dfs_to_keep if k in dfs}

In [290]:

# Limpiamos valores nulos
for nombre_df, df in dfs.items():
    for band_set in ["rhow", "rhown","rtoa"]:
        dfs[nombre_df] = df.dropna()

In [216]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Invierno'
    elif month in [3, 4, 5]:
        return 'Primavera'
    elif month in [6, 7, 8]:
        return 'Verano'
    else:
        return 'Otoño'
    

def get_zone(buoy):
    if buoy in ["CTD1", "CTD2", "CTD3", "CTD4"]:
        return 'Zona-1'
    elif buoy in ["CTD6", "CTD8", "CTD9", "CTD10", "CTD12"]:
        return 'Zona-2'
    elif buoy in ["CTD7"]:
        return 'Zona-3'
    elif buoy in ["CTD11"]:
        return 'Zona-4'

In [291]:

for nombre_df, df in dfs.items():
    # Marcamos las columnas de CHl alta (equivalente a quantile(0.93))
    df["High_Chl"] = df["Chl"]>5
    # Sacamos la estación de cada fecha
    df['Date'] = pd.to_datetime(df['Date'])
    df['Season'] = df['Date'].dt.month.apply(get_season)
    
    # Etiquetamos la zona de la observación (comentado porque para aplicar el modelo habría que segmentar todo el Mar Menor - se puede hacer por px)
    # df['Zone'] = df['Buoy'].apply(get_zone)
    # Ponemos las columnas como categóricas, para Season y Zone
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype('category')

    df = pd.concat([df.drop(columns=["Season"]),pd.get_dummies(df["Season"])], axis=1)
    dfs[nombre_df] = df
    

In [53]:
# Filtro para quitar columnas muy correlacionadas de cada dataset - de momento no lo usamos
def filtrar_columnas(df):

    # 1. Separar columnas numéricas y no numéricas
    df_numericas = df.select_dtypes(include='number')
    df_no_numericas = df.select_dtypes(exclude='number')

    # 2. Calcular la correlación con 'Chl' solo entre columnas numéricas
    correlaciones = df_numericas.corr()['Chl'].drop('Chl')

    # 3. Filtrar predictores numéricos con correlación significativa
    umbral_corr = 0.1
    columnas_utiles = correlaciones[correlaciones.abs() >= umbral_corr].index.tolist()

    # 4. Reconstruir el DataFrame con:
    # - Las columnas numéricas útiles
    # - La columna objetivo 'Chl'
    df_filtrado = pd.concat([df[columnas_utiles + ['Chl']]], axis=1)

    # Calcular la matriz de correlación entre predictores
    corr_matrix = df_filtrado.drop(columns='Chl').corr().abs()

    # Seleccionar columnas a eliminar (altamente correlacionadas entre sí)
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    columnas_redundantes = [col for col in upper.columns if any(upper[col] > 0.98)]

    # Eliminar redundantes
    df_final = pd.concat([df_filtrado.drop(columns=columnas_redundantes), df_no_numericas], axis=1)
    df_final = df_final.drop(columns=["Date", "Buoy"])

    return df_final


for nombre_df, df in dfs.items():
    dfs[nombre_df] = filtrar_columnas(df)

In [218]:
model_params ={
    "XGB" : {
        'n_estimators': 1000,
        'learning_rate': 0.01,
        'max_depth': 7,
        'min_child_weight': 3,
        'subsample': 0.9,
        'colsample_bytree': 0.9,
        'device': 'cpu',
        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'enable_categorical': True,
        #'early_stopping_rounds': 50,
        'eval_metric': 'rmse'
        },

    "LBM" : {
        'learning_rate': 0.05,
        'num_leaves': 20,
        'max_depth': 7,
        'min_child_samples': 3,
        'subsample': 0.6,
        'colsample_bytree': 0.9,
        'n_estimators': 1000,
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'device': 'cpu',  
        'verbosity': -1,
        #'early_stopping_rounds': 50
        },

    "MLP": {
        'hidden_layer_sizes': (100,),
        'activation': 'relu',
        'solver': 'adam',
        'alpha': 0.0001,
        'learning_rate': 'constant',
        'learning_rate_init': 0.001,
        'max_iter': 200,
        'shuffle': True,
        'random_state': None,
        'tol': 1e-4,
        'n_iter_no_change': 25,
        'verbose': False,
        'early_stopping': True,
        'validation_fraction': 0.2
        },

    "SVR": {
        'kernel': 'sigmoid',        
        'C': 1.0,               
        'epsilon': 0.1,           
        'gamma': 'scale',        
        'shrinking': True,
        'tol': 1e-3,
        'max_iter': -1,          
        'verbose': False,
    },

    "KNN": {
        'n_neighbors': 5,
        'weights': 'uniform',      
        'algorithm': 'auto',      
        'leaf_size': 30,
        'p': 2,                    
        'metric': 'minkowski',
        'n_jobs': -1             
    },

    "RF": {
        'n_estimators': 100,         
        'criterion': 'squared_error',
        'max_depth': 10,         
        'min_samples_split': 2,
        'min_samples_leaf': 1,    
        'bootstrap': True,
        'random_state': 42,
        'verbose': 0
    },

    "CAT": {
        'iterations': 1000,
        'learning_rate': 0.03,
        'depth': 6,
        'l2_leaf_reg': 3.0,
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'random_seed': 42,
        'allow_writing_files': False,
        'early_stopping_rounds': 50,
        'verbose': False
    },

    "EN": {
        'alpha': 1.0,              # fuerza de regularización
        'l1_ratio': 0.5,           # mezcla entre L1 (lasso) y L2 (ridge)
        'fit_intercept': True,
        'max_iter': 1000,
        'tol': 1e-4,
        'selection': 'cyclic',
        'random_state': 42
    }

}


models = {
    "XGB": XGBRegressor(**model_params['XGB']),
    "LBM": LGBMRegressor(**model_params['LBM']),
    "MLP": MLPRegressor(**model_params['MLP']),
    #"SVR": SVR(**model_params['SVR']),
    "KNN": KNeighborsRegressor(**model_params['KNN']),
    #"LR": LinearRegression().
    "RF": RandomForestRegressor(**model_params['RF']),
    "CAT": CatBoostRegressor(**model_params["CAT"]),
    "EN":  ElasticNet(**model_params["EN"])
}

In [241]:
results = {}

for nombre_df, df in list(dfs.items()):
    #print(nombre_df)
    df = df.iloc[:,4:]

    # Para usar solamente bandas, sin combinaciones
    # if 'TOA' in nombre_df:
    #     # TOA solamente con las bandas, parece que las combinaciones solo meten ruido
    #     df = df.iloc[:,np.r_[0:14, 58:60]]
    # if 'rhow' in nombre_df and 'rhown' not in nombre_df:
    #     df = df.iloc[:,np.r_[0:9, 53:55]]
    # if 'rhown' in nombre_df:
    #     df = df.iloc[:,np.r_[0:7, 51:53]]

    train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["High_Chl"]) # TEST 20% TRAIN 80%
    target = "Chl"

    train, val = train_test_split(train, test_size=0.25, random_state=42, stratify=train["High_Chl"]) # TRAIN 60% VAL 20% TEST%

    X_train = train.drop(columns=[target,"High_Chl"])
    X_val = val.drop(columns=[target, "High_Chl"])
    X_test = test.drop(columns=[target, "High_Chl"])
    y_train = train[target]
    y_val = val[target]
    y_test = test[target]

    
    scaler_X = RobustScaler()
    scaler_y = RobustScaler()
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_val_scaled = scaler_X.transform(X_val)
    X_test_scaled = scaler_X.transform(X_test)
    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
    
    results[nombre_df] = {name: {'RMSE': None, 'R2': None} for name in models}
    val_preds = {}
    test_preds = {}

    for name, model in models.items():
        print(f"Fitting {name} for {nombre_df}")
        if name in ["MLP", "SVR", "KNN", "LR", "EN"]:
            model.fit(X_train_scaled, y_train_scaled)
            #test_pred = model.predict(X_test_scaled)
            val_pred = scaler_y.inverse_transform(model.predict(X_val_scaled).reshape(-1, 1)).ravel()
            test_pred = scaler_y.inverse_transform(model.predict(X_test_scaled).reshape(-1, 1)).ravel()
        else:
            model.fit(X_train, y_train)
            val_pred = model.predict(X_val)
            test_pred = model.predict(X_test)

        val_preds[name] = val_pred
        test_preds[name] = test_pred

        rmse = np.sqrt(mean_squared_error(y_test, test_pred))
        r2 = r2_score(y_test, test_pred)

        results[nombre_df][name]['RMSE'] = rmse.round(2)
        results[nombre_df][name]['R2'] = r2.round(2)

    # Meta-modelo
    meta_X = np.vstack([val_preds[model] for model in models]).T
    meta_y = y_val.values
    meta_model = Ridge().fit(meta_X, meta_y)

    # Predicción final ensemble
    test_meta_X = np.vstack([test_preds[model] for model in models]).T
    ensemble_pred = meta_model.predict(test_meta_X)

    rmse_ens = np.sqrt(mean_squared_error(y_test, ensemble_pred))
    r2_ens = r2_score(y_test, ensemble_pred)

    results[nombre_df]["Ensemble"] = {
        "RMSE": round(rmse_ens, 2),
        "R2": round(r2_ens, 2)
    }

Fitting XGB for C2RCC_rhow_5x5_depth_lt_1
Fitting LBM for C2RCC_rhow_5x5_depth_lt_1
Fitting MLP for C2RCC_rhow_5x5_depth_lt_1


/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fitting SVR for C2RCC_rhow_5x5_depth_lt_1
Fitting KNN for C2RCC_rhow_5x5_depth_lt_1
Fitting LR for C2RCC_rhow_5x5_depth_lt_1
Fitting RF for C2RCC_rhow_5x5_depth_lt_1
Fitting CAT for C2RCC_rhow_5x5_depth_lt_1
Learning rate set to 0.033272
0:	learn: 3.4129684	total: 2ms	remaining: 2s
1:	learn: 3.3568727	total: 3.27ms	remaining: 1.63s
2:	learn: 3.3074897	total: 4.42ms	remaining: 1.47s
3:	learn: 3.2558681	total: 5.56ms	remaining: 1.39s
4:	learn: 3.2071105	total: 6.7ms	remaining: 1.33s
5:	learn: 3.1585027	total: 8.52ms	remaining: 1.41s
6:	learn: 3.1054135	total: 10.4ms	remaining: 1.48s
7:	learn: 3.0560053	total: 12.1ms	remaining: 1.5s
8:	learn: 3.0141211	total: 13.4ms	remaining: 1.48s
9:	learn: 2.9590940	total: 14.6ms	remaining: 1.45s
10:	learn: 2.9136806	total: 15.9ms	remaining: 1.43s
11:	learn: 2.8789791	total: 17.2ms	remaining: 1.41s
12:	learn: 2.8313709	total: 18.4ms	remaining: 1.4s
13:	learn: 2.7964162	total: 19.6ms	remaining: 1.38s
14:	learn: 2.7483571	total: 20.9ms	remaining: 1.37s
1

/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fitting SVR for C2X-Complex_rhown_3x3_depth_lt_1
Fitting KNN for C2X-Complex_rhown_3x3_depth_lt_1
Fitting LR for C2X-Complex_rhown_3x3_depth_lt_1
Fitting RF for C2X-Complex_rhown_3x3_depth_lt_1
Fitting CAT for C2X-Complex_rhown_3x3_depth_lt_1
Learning rate set to 0.033272
0:	learn: 3.4122952	total: 3.92ms	remaining: 3.92s
1:	learn: 3.3474127	total: 5.18ms	remaining: 2.59s
2:	learn: 3.2991138	total: 6.54ms	remaining: 2.17s
3:	learn: 3.2462580	total: 7.67ms	remaining: 1.91s
4:	learn: 3.1987774	total: 8.84ms	remaining: 1.76s
5:	learn: 3.1550942	total: 10ms	remaining: 1.66s
6:	learn: 3.1008370	total: 11.2ms	remaining: 1.58s
7:	learn: 3.0559803	total: 12.3ms	remaining: 1.52s
8:	learn: 3.0003857	total: 13.4ms	remaining: 1.48s
9:	learn: 2.9573906	total: 14.6ms	remaining: 1.44s
10:	learn: 2.9121269	total: 15.8ms	remaining: 1.42s
11:	learn: 2.8741344	total: 17ms	remaining: 1.4s
12:	learn: 2.8167912	total: 18.2ms	remaining: 1.38s
13:	learn: 2.7756944	total: 19.4ms	remaining: 1.37s
14:	learn: 2.7

...

Fitting SVR for TOA_5x5_depth_lt_1
Fitting KNN for TOA_5x5_depth_lt_1
Fitting LR for TOA_5x5_depth_lt_1
Fitting RF for TOA_5x5_depth_lt_1
Fitting CAT for TOA_5x5_depth_lt_1
Learning rate set to 0.033292
0:	learn: 3.7139214	total: 1.74ms	remaining: 1.74s
1:	learn: 3.6643628	total: 2.85ms	remaining: 1.42s
2:	learn: 3.6006528	total: 3.91ms	remaining: 1.3s
3:	learn: 3.5499114	total: 4.98ms	remaining: 1.24s
4:	learn: 3.4839437	total: 6.08ms	remaining: 1.21s
5:	learn: 3.4257150	total: 7.21ms	remaining: 1.19s
6:	learn: 3.3671876	total: 8.49ms	remaining: 1.2s
7:	learn: 3.3213571	total: 9.72ms	remaining: 1.21s
8:	learn: 3.2791244	total: 10.9ms	remaining: 1.2s
9:	learn: 3.2256604	total: 12ms	remaining: 1.19s
10:	learn: 3.1805125	total: 13.2ms	remaining: 1.19s
11:	learn: 3.1274639	total: 14.3ms	remaining: 1.18s
12:	learn: 3.0864180	total: 15.4ms	remaining: 1.17s
13:	learn: 3.0291593	total: 16.5ms	remaining: 1.16s
14:	learn: 2.9875266	total: 17.6ms	remaining: 1.16s
15:	learn: 2.9318790	total: 18.6

/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fitting SVR for C2RCC_rhown_3x3_depth_lt_1
Fitting KNN for C2RCC_rhown_3x3_depth_lt_1
Fitting LR for C2RCC_rhown_3x3_depth_lt_1
Fitting RF for C2RCC_rhown_3x3_depth_lt_1
Fitting CAT for C2RCC_rhown_3x3_depth_lt_1
Learning rate set to 0.033272
0:	learn: 3.4097600	total: 3.54ms	remaining: 3.54s
1:	learn: 3.3466748	total: 4.68ms	remaining: 2.34s
2:	learn: 3.2995671	total: 5.78ms	remaining: 1.92s
3:	learn: 3.2520073	total: 6.86ms	remaining: 1.71s
4:	learn: 3.1905901	total: 7.93ms	remaining: 1.58s
5:	learn: 3.1424228	total: 9.03ms	remaining: 1.5s
6:	learn: 3.0965614	total: 10.1ms	remaining: 1.44s
7:	learn: 3.0487338	total: 11.2ms	remaining: 1.39s
8:	learn: 2.9968750	total: 12.3ms	remaining: 1.36s
9:	learn: 2.9459715	total: 14.3ms	remaining: 1.41s
10:	learn: 2.8971853	total: 15.8ms	remaining: 1.42s
11:	learn: 2.8647712	total: 16.9ms	remaining: 1.4s
12:	learn: 2.8182326	total: 18.1ms	remaining: 1.38s
13:	learn: 2.7796133	total: 19.3ms	remaining: 1.36s
14:	learn: 2.7328070	total: 20.5ms	remain

/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Fitting SVR for C2RCC_rhow_1x1_depth_lt_1
Fitting KNN for C2RCC_rhow_1x1_depth_lt_1
Fitting LR for C2RCC_rhow_1x1_depth_lt_1
Fitting RF for C2RCC_rhow_1x1_depth_lt_1
Fitting CAT for C2RCC_rhow_1x1_depth_lt_1
Learning rate set to 0.033272
0:	learn: 3.4073191	total: 3.12ms	remaining: 3.12s
1:	learn: 3.3490540	total: 4.36ms	remaining: 2.17s
2:	learn: 3.3029165	total: 5.53ms	remaining: 1.84s
3:	learn: 3.2483306	total: 6.72ms	remaining: 1.67s
4:	learn: 3.2034362	total: 8ms	remaining: 1.59s
5:	learn: 3.1537909	total: 9.87ms	remaining: 1.63s
6:	learn: 3.1054676	total: 11.2ms	remaining: 1.59s
7:	learn: 3.0517937	total: 12.4ms	remaining: 1.53s
8:	learn: 2.9973583	total: 13.6ms	remaining: 1.5s
9:	learn: 2.9458330	total: 14.8ms	remaining: 1.46s
10:	learn: 2.9123639	total: 16ms	remaining: 1.44s
11:	learn: 2.8800820	total: 17.1ms	remaining: 1.41s
12:	learn: 2.8407615	total: 18.3ms	remaining: 1.39s
13:	learn: 2.8112349	total: 19.4ms	remaining: 1.37s
14:	learn: 2.7611414	total: 20.6ms	remaining: 1.35

In [242]:
# Para hacer un dataframe con multiindex en el que se vea todo

rows = []

for df_name, model_scores in results.items():
    row = {}
    for model_name, metrics in model_scores.items():
        for metric_name, value in metrics.items():
            # Clave: (métrica, modelo) → orden correcto para columnas
            row[(metric_name, model_name)] = value
    rows.append((df_name, row))

df_results = pd.DataFrame.from_dict(dict(rows), orient="index")
df_results.columns = pd.MultiIndex.from_tuples(df_results.columns, names=["Metric", "Model"])
df_results = df_results.sort_index(axis=1, level=0)
df_results = df_results.sort_index(axis=0)

In [243]:
df_results

Metric                              R2                                         \
Model                              CAT    EN Ensemble   KNN   LBM    LR   MLP   
C2RCC_rhow_1x1_depth_lt_1         0.60  0.32     0.77  0.65  0.62  0.64  0.78   
C2RCC_rhow_3x3_depth_lt_1         0.62  0.33     0.79  0.63  0.63  0.47  0.78   
C2RCC_rhow_5x5_depth_lt_1         0.67  0.37     0.78  0.64  0.65  0.64  0.80   
C2RCC_rhow_9x9_depth_lt_1         0.78  0.37     0.69  0.62  0.70  0.49  0.77   
C2RCC_rhown_1x1_depth_lt_1        0.51  0.30     0.72  0.68  0.62  0.44  0.77   
C2RCC_rhown_3x3_depth_lt_1        0.62  0.31     0.78  0.61  0.62  0.48  0.76   
C2RCC_rhown_5x5_depth_lt_1        0.64  0.36     0.69  0.67  0.64  0.62  0.81   
C2RCC_rhown_9x9_depth_lt_1        0.81  0.36     0.74  0.62  0.70  0.55  0.77   
C2X-Complex_rhow_1x1_depth_lt_1   0.68  0.36     0.43  0.64  0.70  0.15  0.72   
C2X-Complex_rhow_3x3_depth_lt_1   0.71  0.37     0.51  0.60  0.56  0.10  0.72   
C2X-Complex_rhow_5x5_depth_lt_1   0.71  0.41     0.63  0.58  0.67 -2.23  0.73   
C2X-Complex_rhow_9x9_depth_lt_1   0.80  0.50     0.74  0.64  0.81  0.41  0.77   
C2X-Complex_rhown_1x1_depth_lt_1  0.73  0.40     0.77  0.69  0.68  0.55  0.72   
C2X-Complex_rhown_3x3_depth_lt_1  0.67  0.37     0.53  0.56  0.60  0.54  0.72   
C2X-Complex_rhown_5x5_depth_lt_1  0.71  0.44     0.78  0.62  0.65  0.49  0.72   
C2X-Complex_rhown_9x9_depth_lt_1  0.79  0.50     0.79  0.67  0.74  0.45  0.80   
C2X_rhow_1x1_depth_lt_1           0.68  0.62     0.74  0.60  0.59  0.08  0.55   
C2X_rhow_3x3_depth_lt_1           0.62  0.47     0.52  0.62  0.59 -0.88  0.65   
C2X_rhow_5x5_depth_lt_1           0.48  0.54     0.69  0.57  0.50  0.54  0.76   
C2X_rhow_9x9_depth_lt_1           0.68  0.65     0.70  0.70  0.67  0.37  0.79   
C2X_rhown_1x1_depth_lt_1          0.61  0.53     0.59  0.50  0.52  0.33  0.53   
C2X_rhown_3x3_depth_lt_1          0.61  0.32     0.69  0.51  0.60  0.03  0.66   
C2X_rhown_5x5_depth_lt_1          0.50  0.43     0.60  0.45  0.43  0.72  0.71   
C2X_rhown_9x9_depth_lt_1          0.68  0.60     0.76  0.55  0.59  0.47  0.77   
TOA_1x1_depth_lt_1                0.36  0.01     0.24  0.37  0.45  0.27  0.41   
TOA_3x3_depth_lt_1                0.63  0.02     0.45  0.42  0.51  0.45  0.50   
TOA_5x5_depth_lt_1                0.60  0.04     0.49  0.46  0.42  0.38  0.57   
TOA_9x9_depth_lt_1                0.60  0.04     0.42  0.46  0.54 -0.03  0.68   

Metric                                              RMSE                       \
Model                               RF   SVR   XGB   CAT    EN Ensemble   KNN   
C2RCC_rhow_1x1_depth_lt_1         0.62  0.30  0.39  3.00  3.93     2.28  2.82   
C2RCC_rhow_3x3_depth_lt_1         0.61  0.33  0.58  2.94  3.92     2.20  2.90   
C2RCC_rhow_5x5_depth_lt_1         0.70  0.35  0.78  2.72  3.79     2.23  2.86   
C2RCC_rhow_9x9_depth_lt_1         0.71  0.33  0.65  2.26  3.80     2.66  2.96   
C2RCC_rhown_1x1_depth_lt_1        0.56  0.30  0.58  3.33  3.99     2.54  2.70   
C2RCC_rhown_3x3_depth_lt_1        0.59  0.32  0.76  2.94  3.96     2.25  2.96   
C2RCC_rhown_5x5_depth_lt_1        0.71  0.35  0.77  2.85  3.82     2.65  2.74   
C2RCC_rhown_9x9_depth_lt_1        0.76  0.33  0.82  2.07  3.82     2.43  2.93   
C2X-Complex_rhow_1x1_depth_lt_1   0.60  0.23  0.66  2.71  3.81     3.59  2.87   
C2X-Complex_rhow_3x3_depth_lt_1   0.62  0.28  0.59  2.57  3.79     3.33  3.03   
C2X-Complex_rhow_5x5_depth_lt_1   0.58  0.29  0.59  2.56  3.65     2.90  3.09   
C2X-Complex_rhow_9x9_depth_lt_1   0.68  0.22  0.53  2.14  3.38     2.43  2.84   
C2X-Complex_rhown_1x1_depth_lt_1  0.59  0.33  0.61  2.49  3.69     2.27  2.65   
C2X-Complex_rhown_3x3_depth_lt_1  0.53  0.36  0.64  2.75  3.77     3.27  3.18   
C2X-Complex_rhown_5x5_depth_lt_1  0.65  0.32  0.73  2.55  3.58     2.23  2.94   
C2X-Complex_rhown_9x9_depth_lt_1  0.69  0.29  0.53  2.17  3.38     2.17  2.74   
C2X_rhow_1x1_depth_lt_1           0.62  0.23  0.60  2.68  2.94     2.45  3.01   
C2X_rhow_3x3_depth_lt_1       

In [114]:
df_results

metric                              r2              rmse            
model                              LBM   MLP   XGB   LBM   MLP   XGB
C2RCC_rhow_5x5_depth_lt_1         0.51  0.67  0.72  2.59  2.12  1.95
C2X-Complex_rhown_3x3_depth_lt_1  0.52  0.63  0.70  2.57  2.25  2.03
TOA_9x9_depth_lt_1                0.50  0.48  0.61  2.61  2.65  2.29
C2X_rhow_3x3_depth_lt_1           0.54  0.64  0.64  2.50  2.23  2.24
C2RCC_rhow_9x9_depth_lt_1         0.63  0.75  0.57  2.25  1.87  2.44
C2X-Complex_rhown_5x5_depth_lt_1  0.63  0.55  0.73  2.25  2.48  1.92
C2X_rhown_9x9_depth_lt_1          0.49  0.67  0.44  2.66  2.13  2.77
TOA_3x3_depth_lt_1                0.56  0.47  0.57  2.44  2.69  2.41
C2X-Complex_rhown_9x9_depth_lt_1  0.58  0.71  0.64  2.39  2.01  2.22
C2X-Complex_rhow_3x3_depth_lt_1   0.66  0.24  0.68  2.15  3.24  2.10
C2X_rhow_1x1_depth_lt_1           0.27  0.52  0.45  3.16  2.58  2.74
C2RCC_rhown_1x1_depth_lt_1        0.41  0.58  0.68  2.85  2.41  2.11
C2X_rhown_1x1_depth_lt_1          0.20  0.35  0.16  3.31  2.99  3.40
C2RCC_rhown_9x9_depth_lt_1        0.45  0.67  0.54  2.74  2.13  2.53
C2X_rhow_5x5_depth_lt_1           0.52  0.65  0.68  2.56  2.19  2.09
C2X_rhown_5x5_depth_lt_1          0.51  0.38  0.59  2.58  2.91  2.36
C2X_rhow_9x9_depth_lt_1           0.60  0.69  0.67  2.34  2.07  2.14
C2X_rhown_3x3_depth_lt_1          0.10  0.57  0.58  3.51  2.43  2.40
C2RCC_rhown_5x5_depth_lt_1        0.68  0.64  0.79  2.10  2.22  1.70
TOA_1x1_depth_lt_1                0.66  0.47  0.61  2.13  2.68  2.29
C2X-Complex_rhow_5x5_depth_lt_1   0.88  0.03  0.88  1.29  3.65  1.27
C2X-Complex_rhow_9x9_depth_lt_1   0.61  0.13  0.68  2.30  3.46  2.10
C2X-Complex_rhow_1x1_depth_lt_1   0.66  0.57  0.67  2.18  2.42  2.14
C2X-Complex_rhown_1x1_depth_lt_1  0.43  0.68  0.51  2.80  2.11  2.59
C2RCC_rhow_3x3_depth_lt_1         0.64  0.63  0.56  2.24  2.26  2.47
TOA_5x5_depth_lt_1                0.52  0.48  0.55  2.54  2.66  2.48
C2RCC_rhown_3x3_depth_lt_1        0.70  0.65  0.64  2.03  2.19  2.22
C2RCC_rhow_1x1_depth_lt_1         0.23  0.55  0.64  3.25  2.50  2.22

In [292]:
def objective(trial, df, target, model_name):
    df = df.iloc[:,4:]
    # Mismos splits que para el entrenamiento
    train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["High_Chl"]) # TEST 20% TRAIN 80%
    target = "Chl"
    train, val = train_test_split(train, test_size=0.25, random_state=42, stratify=train["High_Chl"])

    X = train.drop(columns=[target, 'High_Chl'])
    y = train[target]
    y_class = train["High_Chl"]

    # Definimos los folds
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    oof_preds = np.zeros(len(train))  # Almacenar las predicciones OOF

    start = time.time()
    
    if model_name == "LBM":
        params = {
            'objective': 'regression',
            'metric': 'rmse',
            'boosting_type': 'gbdt',
            'device': 'cpu',  # usa CPU/GPU
            'verbosity': -1,
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'num_leaves': trial.suggest_categorical('num_leaves', [20, 40, 60, 80]),
            'max_depth': trial.suggest_int('max_depth', 5, 8),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 25),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'n_estimators': trial.suggest_categorical('n_estimators', [500, 1000, 2000])
        }

    if model_name == "XGB":
        params = {
            'n_estimators': trial.suggest_categorical('n_estimators', [500, 1000, 2000]),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 5, 8),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 4),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'device': 'cpu',
            'objective': 'reg:squarederror',
            'tree_method': 'hist',
            'enable_categorical': True,
            'eval_metric': 'rmse'
        }
    
    if model_name == "MLP":
        # Así para evitar warning de definir como tuple
        hidden_options = {
            '50': (50,),
            '100': (100,),
            '100_50': (100, 50),
            '128_64': (128, 64)
        }
        key = trial.suggest_categorical('hidden_layer_sizes', list(hidden_options.keys()))
        params = {
            'hidden_layer_sizes': hidden_options[key],
            #'hidden_layer_sizes': trial.suggest_categorical('hidden_layer_sizes', [(50,), (100,), (100, 50), (128, 64)]),
            'activation': trial.suggest_categorical('activation', ['relu', 'tanh']),
            'solver': trial.suggest_categorical('solver', ['adam', 'sgd']),
            'alpha': trial.suggest_float('alpha', 1e-5, 1e-1, log=True),
            'learning_rate': trial.suggest_categorical('learning_rate', ['constant', 'adaptive']),
            'learning_rate_init': trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
            'max_iter': 200,
            'n_iter_no_change': 25,
            'early_stopping': True,
            'validation_fraction': 0.2,
            'random_state': 42,
            'verbose': False
        }

    if model_name == "SVR":
        params = {
            'kernel': trial.suggest_categorical('kernel', ['rbf', 'sigmoid']),
            'C': trial.suggest_float('C', 0.1, 10.0, log=True),
            'epsilon': trial.suggest_float('epsilon', 0.01, 0.2),
            'gamma': trial.suggest_categorical('gamma', ['scale', 'auto']),
            'shrinking': True,
            'tol': 1e-3,
            'max_iter': -1,
            'verbose': False
        }

    if model_name == "KNN":
        params = {
            'n_neighbors': trial.suggest_int('n_neighbors', 3, 15),
            'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
            'algorithm': 'auto',
            'leaf_size': trial.suggest_int('leaf_size', 10, 40),
            'p': 2,  # 1 = manhattan, 2 = euclídea
            'metric': 'minkowski',
            'n_jobs': -1
        }

    if model_name == "LR":
        # No sirve de mucho, pero por completitud
        params = {
            'fit_intercept': trial.suggest_categorical('fit_intercept', [True, False]),
            'positive': trial.suggest_categorical('positive', [True, False]),
        }

    if model_name == "RF":
        params = {
            'n_estimators': trial.suggest_categorical('n_estimators', [100, 300, 500]),
            'max_depth': trial.suggest_int('max_depth', 5, 15),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
            'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
            'random_state': 42,
            'verbose': 0
        }

    if model_name == "CAT":
        params = {
            'iterations': trial.suggest_categorical('iterations', [500, 1000, 2000]),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'depth': trial.suggest_int('depth', 4, 10),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
            'loss_function': 'RMSE',
            'eval_metric': 'RMSE',
            'random_seed': 42,
            'early_stopping_rounds': 50,
            'verbose': False,
        }

    if model_name == "EN":
        params = {
            'alpha': trial.suggest_float('alpha', 1e-4, 10.0, log=True),
            'l1_ratio': trial.suggest_float('l1_ratio', 0.0, 1.0),
            'fit_intercept': True,
            'max_iter': 1000,
            'tol': 1e-4,
            'selection': 'cyclic',
            'random_state': 42
        }

    # Cargamos el modelo correspondiente
    model = models[model_name](**params)

    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
        print(f"Fold {fold+1}")
        
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        if model_name in ["MLP", "SVR", "KNN", "LR", "EN"]:
            scaler_X = RobustScaler()
            scaler_y = RobustScaler()
            X_train_scaled = scaler_X.fit_transform(X_train)
            X_val_scaled = scaler_X.transform(X_val)
            y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
            model.fit(X_train_scaled, y_train_scaled)
            y_pred = scaler_y.inverse_transform(model.predict(X_val_scaled).reshape(-1, 1)).ravel()
        else:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
        
        oof_preds[val_idx] = y_pred


    print(f"Running time: {time.time() - start:.1f} sec")
    # Calculamos el RMSE OOF
    rmse_score = np.sqrt(mean_squared_error(y, oof_preds))
    r2 = r2_score(y, oof_preds)
    print(f"OOF RMSE: {rmse_score:.2f} | R2: {r2:.2f}")
    
    return r2

In [293]:
models = {
    "XGB": XGBRegressor,
    "LBM": LGBMRegressor,
    "MLP": MLPRegressor,
    "SVR": SVR,
    "KNN": KNeighborsRegressor,
    "LR": LinearRegression,
    "RF": RandomForestRegressor,
    "CAT": CatBoostRegressor,
    "EN":  ElasticNet
}

def run_optuna(df, target, n_trials, model_name):
    results = {}

    print(f"Buscando mejores hiperparámetros para {model_name}...")
    study = optuna.create_study(direction='maximize')
    study.optimize(lambda trial: objective(trial, df, target, model_name), n_trials=n_trials, timeout= 600)
    
    print(f"\n✅ {model_name} - Mejor R2: {study.best_value:.2f}")
    print(f"📋 Parámetros: {study.best_params}\n")
    
    results[model_name] = {
        'best_params': study.best_params,
        'best_score': study.best_value,
        'study': study
    }
    return results

#df = dfs["C2X_rhown_3x3_depth_lt_1"]

# all_results = {}
n_trials = 25
# for model_name in models.keys():
#     all_results[model_name] = run_optuna(df, "Chl", n_trials, model_name)
global_results = {}

for df_name, df in dfs.items():
    print(f"🔍 Optimizando en {df_name}...")
    for model_name in models.keys():
        key = (df_name, model_name)
        result = run_optuna(df, "Chl", n_trials, model_name)
        global_results[key] = result[model_name]

[I 2025-07-11 16:08:52,346] A new study created in memory with name: no-name-b643c892-28af-4ffa-a3d5-07df1c04bcc9


🔍 Optimizando en C2RCC_rhow_5x5_depth_lt_1...
Buscando mejores hiperparámetros para XGB...
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2025-07-11 16:09:01,324] Trial 0 finished with value: 0.6786606658483075 and parameters: {'n_estimators': 1000, 'learning_rate': 0.03363059712784438, 'max_depth': 5, 'min_child_weight': 2, 'subsample': 0.7354620669213784, 'colsample_bytree': 0.926760985439751}. Best is trial 0 with value: 0.6786606658483075.


Running time: 9.0 sec
OOF RMSE: 1.97 | R2: 0.68
Fold 1


In [ ]:

with open("global_results.pkl", "wb") as f:
    pickle.dump(global_results, f)


In [286]:
global_results

{('C2RCC_rhow_5x5_depth_lt_1',
  'RF'): {'best_params': {'n_estimators': 500,
   'max_depth': 15,
   'min_samples_split': 4,
   'min_samples_leaf': 3,
   'bootstrap': False}, 'best_score': 0.6353231990874344, 'study': <optuna.study.study.Study at 0x76963bb04760>},
 ('C2RCC_rhow_5x5_depth_lt_1',
  'EN'): {'best_params': {'alpha': 2.0185768774268875,
   'l1_ratio': 0.17644243451557984}, 'best_score': 0.4064433021731192, 'study': <optuna.study.study.Study at 0x76963ba5db50>},
 ('C2X-Complex_rhown_3x3_depth_lt_1',
  'RF'): {'best_params': {'n_estimators': 500,
   'max_depth': 7,
   'min_samples_split': 6,
   'min_samples_leaf': 2,
   'bootstrap': False}, 'best_score': 0.4824625378062575, 'study': <optuna.study.study.Study at 0x76963b9c29a0>},
 ('C2X-Complex_rhown_3x3_depth_lt_1',
  'EN'): {'best_params': {'alpha': 0.4923404321333851,
   'l1_ratio': 0.0721806154716893}, 'best_score': 0.49010994246426043, 'study': <optuna.study.study.Study at 0x76963ba2e5e0>},
 ('TOA_9x9_depth_lt_1',
  'RF')

In [287]:
from collections import defaultdict
import pandas as pd

# Recolectar todos los mejores parámetros por modelo
param_values = defaultdict(list)

for (df_name, model_name), result in global_results.items():
    param_values[model_name].append(result['best_params'])

# Convertir cada grupo de parámetros en DataFrame
avg_params = {}

for model_name, param_list in param_values.items():
    df_params = pd.DataFrame(param_list)
    # Calcular media solo de los parámetros numéricos
    df_mean = df_params.select_dtypes(include='number').mean().to_dict()
    
    # Para categóricos: tomamos la moda (más frecuente)
    df_cat = df_params.select_dtypes(exclude='number').mode().iloc[0].to_dict()
    
    # Juntamos ambos
    avg_params[model_name] = {**df_mean, **df_cat}


IndexError: single positional indexer is out-of-bounds

In [55]:
model_names = ["XGB", "LBM"]
n_trials = 20
results = run_optuna(train, "Chl", n_trials, model_names)

[I 2025-06-24 12:22:47,253] A new study created in memory with name: no-name-fa720214-6f8c-4099-99f4-edf6841d7c5d


Buscando mejores hiperparámetros para XGB...
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2025-06-24 12:22:49,621] Trial 0 finished with value: 2.6402744486891083 and parameters: {'n_estimators': 1000, 'learning_rate': 0.010250582517387313, 'max_depth': 6, 'min_child_weight': 2, 'subsample': 0.6149501746489763, 'colsample_bytree': 0.612993571753984}. Best is trial 0 with value: 2.6402744486891083.


Running time: 2.4 sec
OOF RMSE: 2.6403
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2025-06-24 12:22:54,496] Trial 1 finished with value: 2.6551140502554027 and parameters: {'n_estimators': 1000, 'learning_rate': 0.006437532229785393, 'max_depth': 8, 'min_child_weight': 3, 'subsample': 0.9780500815920048, 'colsample_bytree': 0.7647705570506433}. Best is trial 0 with value: 2.6402744486891083.


...

Fold 5
Running time: 0.3 sec
OOF RMSE: 2.6784
Fold 1
Fold 2
Fold 3


[I 2025-06-24 12:24:09,456] Trial 18 finished with value: 2.7875795003296453 and parameters: {'learning_rate': 0.005596380240543389, 'num_leaves': 20, 'max_depth': 7, 'min_child_samples': 15, 'subsample': 0.9879610090915855, 'colsample_bytree': 0.8125020591488976, 'n_estimators': 1000}. Best is trial 11 with value: 2.6523885863923535.


Fold 4
Fold 5
Running time: 0.5 sec
OOF RMSE: 2.7876
Fold 1
Fold 2
Fold 3
Fold 4


[I 2025-06-24 12:24:09,798] Trial 19 finished with value: 2.6784534814223617 and parameters: {'learning_rate': 0.015045130475019163, 'num_leaves': 20, 'max_depth': 6, 'min_child_samples': 27, 'subsample': 0.8784043880020499, 'colsample_bytree': 0.855483063357705, 'n_estimators': 1000}. Best is trial 11 with value: 2.6523885863923535.


Fold 5
Running time: 0.3 sec
OOF RMSE: 2.6785

✅ LBM - Mejor RMSLE: 2.6524
📋 Parámetros: {'learning_rate': 0.08933589129532095, 'num_leaves': 20, 'max_depth': 7, 'min_child_samples': 27, 'subsample': 0.6011289390155303, 'colsample_bytree': 0.8961713183846663, 'n_estimators': 2000}



In [56]:
results

{'XGB': {'best_params': {'n_estimators': 2000,
   'learning_rate': 0.005038520390145823,
   'max_depth': 8,
   'min_child_weight': 1,
   'subsample': 0.8658250277664441,
   'colsample_bytree': 0.907105544104575},
  'best_score': 2.5861375238851174,
  'study': <optuna.study.study.Study at 0x705cdee95c70>},
 'LBM': {'best_params': {'learning_rate': 0.08933589129532095,
   'num_leaves': 20,
   'max_depth': 7,
   'min_child_samples': 27,
   'subsample': 0.6011289390155303,
   'colsample_bytree': 0.8961713183846663,
   'n_estimators': 2000},
  'best_score': 2.6523885863923535,
  'study': <optuna.study.study.Study at 0x705cdeddeb50>}}

In [30]:
#best_params = {'n_estimators': 2000, 'learning_rate': 0.04814113153747816, 'max_depth': 7, 'min_child_weight': 1, 'subsample': 0.9437728415677074, 'colsample_bytree': 0.909668290806239}
# {'n_estimators': 1000, 'learning_rate': 0.010767242978932143, 'max_depth': 7, 'min_child_weight': 1, 'subsample': 0.8684177755167737, 'colsample_bytree': 0.7303912217157997}
# {'n_estimators': 1000, 'learning_rate': 0.014410895948871282, 'max_depth': 8, 'min_child_weight': 1, 'subsample': 0.9990473761909979, 'colsample_bytree': 0.86335191588344}

#{'learning_rate': 0.08933589129532095, 'num_leaves': 20, 'max_depth': 7, 'min_child_samples': 27, 'subsample': 0.6011289390155303, 'colsample_bytree': 0.8961713183846663, 'n_estimators': 2000}
best_params_xgb = {'n_estimators': 1000, 'learning_rate': 0.012, 'max_depth': 7, 'min_child_weight': 1, 'subsample': 0.95, 'colsample_bytree': 0.85}
best_params_lbm = {'learning_rate': 0.089, 'num_leaves': 20, 'max_depth': 7, 'min_child_samples': 27, 'subsample': 0.60, 'colsample_bytree': 0.89, 'n_estimators': 2000}


In [18]:
params_xgb= {
    'n_estimators': 1000,
    'learning_rate': 0.012,
    'max_depth': 7,
    'min_child_weight': 1,
    'subsample': 0.95,
    'colsample_bytree': 0.85,
    'device': 'cpu',
    'objective': 'reg:squarederror',
    'tree_method': 'hist',
    'enable_categorical': True,
    'early_stopping_rounds': 50,
    'eval_metric': 'rmse'}

params_lbm = {
    'learning_rate': 0.089,
    'num_leaves': 20,
    'max_depth': 7,
    'min_child_samples': 27,
    'subsample': 0.6,
    'colsample_bytree': 0.89,
    'n_estimators': 2000,
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'device': 'cpu',  
    'verbosity': -1,
    'early_stopping_rounds': 50}

### Entrenamiento XGB y LBM

In [41]:
save_folder = "training_results"
FOLDS = 5
kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)

models = {
    'LightGBM': LGBMRegressor(**params_lbm),
    'XGBoost': XGBRegressor(**params_xgb)
}
models_storage = {name: [] for name in models}

results = {name: {'oof': np.zeros(len(train)), 'pred': np.zeros(len(test)), 'rmse': [], 'r2': [], 'r2_tfg' : []} for name in models}

target = "Chl"
X = train.drop(columns=[target, 'High_Chl'])
X_test = test.drop(columns=[target, 'High_Chl'])
y = train[target]
y_class = train["High_Chl"]

    
# Definimos los folds
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print(f"\n=== Training {name} ===")
    for i, (train_idx, valid_idx) in enumerate(skf.split(X, y_class)):
        print(f"\nFold {i+1}")
        x_train, y_train = X.iloc[train_idx], y[train_idx]
        x_valid, y_valid = X.iloc[valid_idx], y[valid_idx]
        
        start = time.time()
        
        if name == 'XGBoost':
            model.fit(x_train, y_train, eval_set=[(x_valid, y_valid)], verbose=500)
        elif name == 'LightGBM':
            model.fit(x_train, y_train, eval_set=[(x_valid, y_valid)])

        models_storage[name].append(model)
        oof_pred = model.predict(x_valid)
        test_pred = model.predict(X_test)
        
        results[name]['oof'][valid_idx] = oof_pred
        results[name]['pred'] += test_pred / FOLDS
        
        rmse = np.sqrt(mean_squared_error(y_valid, oof_pred))
        r2 = r2_score(y_valid, oof_pred)

        results[name]['rmse'].append(rmse)
        results[name]['r2'].append(r2)

        
        print(f"Fold {i+1} RMSE: {rmse:.2f} | R2: {r2:.2f} | R2_TFG {r2_corr:.2f}")
        print(f"Training time: {time.time() - start:.1f} sec")
    


print("\n=== Model Comparison ===")
for name in models:
    mean_rmse = np.mean(results[name]['rmse'])
    std_rmse = np.std(results[name]['rmse'])
    print(f"{name} - Mean RMSE: {mean_rmse:.2f} ± {std_rmse:.2f}")
    mean_r2 = np.mean(results[name]['r2'])
    std_r2 = np.std(results[name]['r2'])
    print(f"{name} - Mean R2: {mean_r2:.2f} ± {std_r2:.2f}")

    


=== Training LightGBM ===

Fold 1
Fold 1 RMSE: 2.99 | R2: 0.48 | R2_TFG 0.53
Training time: 0.0 sec

Fold 2
Fold 2 RMSE: 2.44 | R2: 0.35 | R2_TFG 0.37
Training time: 0.0 sec

Fold 3
Fold 3 RMSE: 2.89 | R2: 0.50 | R2_TFG 0.58
Training time: 0.0 sec

Fold 4
Fold 4 RMSE: 2.41 | R2: 0.57 | R2_TFG 0.58
Training time: 0.1 sec

Fold 5
Fold 5 RMSE: 2.58 | R2: 0.35 | R2_TFG 0.36
Training time: 0.0 sec

=== Training XGBoost ===

Fold 1
[0]	validation_0-rmse:4.14220
[427]	validation_0-rmse:3.18872
Fold 1 RMSE: 3.19 | R2: 0.41 | R2_TFG 0.42
Training time: 3.7 sec

Fold 2
[0]	validation_0-rmse:3.04737
[123]	validation_0-rmse:2.57395
Fold 2 RMSE: 2.53 | R2: 0.31 | R2_TFG 0.31
Training time: 1.3 sec

Fold 3
[0]	validation_0-rmse:4.08089
[500]	validation_0-rmse:2.66474
[999]	validation_0-rmse:2.65590
Fold 3 RMSE: 2.66 | R2: 0.58 | R2_TFG 0.65
Training time: 8.5 sec

Fold 4
[0]	validation_0-rmse:3.65318
[500]	validation_0-rmse:1.98663
[999]	validation_0-rmse:1.97681
Fold 4 RMSE: 1.98 | R2: 0.71 | R2_T

In [60]:
xgb_preds = np.load("training_results/XGBoost_pred.npy")
y_test = test[target]

rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))
r2 = r2_score(y_test, xgb_preds)

print(f"XGB - Test RMSE: {rmse:.2f} | Test R2: {r2:.2f}")

lbm_preds = np.load("training_results/LightGBM_pred.npy")
y_test = test[target]

rmse = np.sqrt(mean_squared_error(y_test, lbm_preds))
r2 = r2_score(y_test, lbm_preds)

print(f"LBM - Test RMSE: {rmse:.2f} | Test R2: {r2:.2f}")

XGB - Test RMSE: 3.20 | Test R2: 0.54
LBM - Test RMSE: 3.48 | Test R2: 0.46


In [61]:
oof_xgb = np.load("training_results/XGBoost_oof.npy")
oof_lbm = np.load("training_results/LightGBM_oof.npy")
#oof_nn = np.load("NN_oof.npy")
pred_xgb = np.load("training_results/XGBoost_pred.npy")
pred_lbm = np.load("training_results/LightGBM_pred.npy")
#pred_nn = np.load("NN_pred.npy")

In [62]:
results = {
    'XGB': {'oof': oof_xgb, 'pred': pred_xgb},
    'LBM': {'oof': oof_lbm, 'pred': pred_lbm},   
    #'MLP': {'oof': oof_nn, 'pred': pred_nn},
}

In [64]:
oof_preds = {name: results[name]['oof'] for name in results}
test_preds = {name: results[name]['pred'] for name in results}
y_true = y

In [65]:
meta_X = np.vstack([oof_preds["XGB"], oof_preds["LBM"]]).T
meta_y = y_true  # target original

# Luego entrenar un meta-modelo
from sklearn.linear_model import Ridge
meta_model = Ridge().fit(meta_X, meta_y)
rmse = np.sqrt(mean_squared_error(meta_y, meta_model.predict(meta_X)))
r2 = r2_score(meta_y, meta_model.predict(meta_X))
print(f"RMSE: {rmse:.2f} | R2: {r2:.2f}")

RMSE: 2.55 | R2: 0.52


In [66]:
meta_X_test = np.vstack([test_preds["XGB"], test_preds["LBM"]]).T
test_pred = meta_model.predict(meta_X_test)
rmse = np.sqrt(mean_squared_error(y_test, test_pred))
r2 = r2_score(y_test, test_pred)
print(f"RMSE: {rmse:.2f} | R2: {r2:.2f}")

RMSE: 3.22 | R2: 0.54


### TabPFN

In [1]:
from tabpfn import TabPFNRegressor  

ImportError: cannot import name 'TabPFNRegressor' from 'tabpfn' (/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/tabpfn/__init__.py)

In [69]:
X_train = train.drop(columns=[target, 'High_Chl'])
X_test = test.drop(columns=[target, 'High_Chl'])
y = train[target]
y_test = test[target]

,rhow_B1,rhow_B2,rhow_B4,rhow_B5,Band_15,dif_norm_rhow_B2_B3,dif_inv_rhow_B2_B3,dif_norm_rhow_B2_B4,dif_inv_rhow_B2_B4,dif_norm_rhow_B3_B4,...,dif_rel_4bands_rhow_B3_B4_B5_B2,sum_norm_3bands_rhow_B2_B4_B3,sum_norm_3bands_rhow_B3_B5_B4,dall_gitelson_rhown_B3_B4_B5,dif_norm_4_bands_rhown_B4_B5_B2_B3,sum_norm_3bands_rhown_B3_B5_B4,Chl,High_Chl,Season,Zone
0,0.020469,0.031289,0.011193,0.007345,0.147099,-0.079,4.681,0.473,-57.385,0.532,...,3.040,0.625,0.920,-0.466,0.054,0.923,0.4995,False,Verano,Zona-4
1,0.009607,0.021231,0.010398,0.007248,0.115210,-0.313,22.437,0.342,-49.068,0.592,...,3.558,0.512,0.938,-0.527,0.049,0.940,1.6950,False,Primavera,Zona-3
2,0.009486,0.020363,0.015713,0.011596,0.166120,-0.352,25.583,0.129,-14.534,0.460,...,2.136,0.574,0.929,-0.478,0.061,0.934,2.9870,False,Primavera,Zona-3
3,0.004488,0.011016,0.008279,0.006037,0.088587,-0.372,49.260,0.142,-30.012,0.488,...,2.361,0.550,0.931,-0.483,0.062,0.932,2.4695,False,Primavera,Zona-1
4,0.001168,0.003506,0.001086,0.000560,0.000027,-0.323,139.403,0.527,-635.927,0.727,...,6.156,0.443,0.934,-0.484,0.042,0.944,0.6580,False,Otoño,Zona-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
331,0.006617,0.012153,0.002676,0.001761,0.035969,-0.001,0.211,0.639,-291.347,0.640,...,4.408,0.609,0.938,-0.556,0.031,0.948,0.5110,False,Primavera,Zona-2
332,0.010938,0.015758,0.009683,0.007627,0.216871,-0.166,18.093,0.239,-39.821,0.390,...,1.793,0.673,0.935,-0.445,0.053,0.937,0.4490,False,Otoño,Zona-2
333,0.001308,0.004138,0.001208,0.000618,0.000027,-0.327,119.011,0.548,-585.858,0.742,...,6.597,0.435,0.937,-0.488,0.039,0.948,0.3030,False,Primavera,Zona-1
334,0.003544,0.010456,0.027829,0.024688,0.332638,-0.555,68.239,-0.454,59.704,0.135,...,-1.050,0.815,0.951,-0.257,0.060,0.955,4.3660,False,Otoño,Zona-2


In [ ]:
# Initialize the regressor
regressor = TabPFNRegressor()  
regressor.fit(X_train, y_train)

# Predict on the test set
predictions = regressor.predict(X_test)

# Evaluate the model

rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("RMSE):", rmse)
print("R² Score:", r2)